In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, auc, average_precision_score,
                             precision_score, recall_score, f1_score, accuracy_score,
                             confusion_matrix)

df = pd.read_csv("daily_features.csv")  # exists on main

# clean & sort by time
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').dropna(subset=['date'])

# make a proxy label (top 2% cost per day is "anomalous")
g = df.groupby('date')['cost_sum']
cut = g.transform(lambda s: s.quantile(0.98))
df['label'] = (df['cost_sum'] >= cut).astype(int)

# feature sets
feat6 = [c for c in ['req_count','cost_sum','cost_mean','data_sum','ratio_cost_per_req','ratio_data_per_req'] if c in df.columns]
if len(feat6) < 3:
    # fall back: use what we know we have
    feat6 = [c for c in ['req_count','cost_sum','cost_mean','data_sum'] if c in df.columns]
feat3 = [c for c in ['req_count','cost_sum','cost_mean'] if c in df.columns]

# chronological 80/20 split
cut_idx = int(len(df) * 0.80)
train, valid = df.iloc[:cut_idx], df.iloc[cut_idx:]

Xtr6, Xva6 = train[feat6].to_numpy(), valid[feat6].to_numpy()
Xtr3, Xva3 = train[feat3].to_numpy(), valid[feat3].to_numpy()
ytr, yva   = train['label'].to_numpy(), valid['label'].to_numpy()

def eval_model(clf, Xtr, ytr, Xva, yva, name):
    clf.fit(Xtr, ytr)
    if hasattr(clf, "predict_proba"):
        s = clf.predict_proba(Xva)[:, 1]
    else:
        s = clf.decision_function(Xva)
    p = (s >= 0.5).astype(int)
    pr, rc, _ = precision_recall_curve(yva, s)
    return {
        'Model': name,
        'PR-AUC': auc(rc, pr),
        'AP': average_precision_score(yva, s),
        'Precision': precision_score(yva, p, zero_division=0),
        'Recall': recall_score(yva, p, zero_division=0),
        'F1': f1_score(yva, p, zero_division=0),
        'Accuracy': accuracy_score(yva, p),
        'Confusion Matrix': confusion_matrix(yva, p)
    }, s

results = []

# Baseline: GaussianNB with "full" features (up to 6)
gnb_full = GaussianNB()
res_full, scores_full = eval_model(gnb_full, Xtr6, ytr, Xva6, yva, f"GNB-{len(feat6)}f")
results.append(res_full)

# Reduced: GaussianNB with 3 features
gnb_reduced = GaussianNB()
res_red, scores_red = eval_model(gnb_reduced, Xtr3, ytr, Xva3, yva, "GNB-3f")
results.append(res_red)

# One more classifier: Logistic Regression (on 3 features to keep it simple)
logreg = LogisticRegression(max_iter=500)
res_lr, scores_lr = eval_model(logreg, Xtr3, ytr, Xva3, yva, "LogReg-3f")
results.append(res_lr)

pd.DataFrame([{k:v for k,v in r.items() if k!='Confusion Matrix'} for r in results])


In [ ]:
# pick best by AP
best = max(results, key=lambda d: d['AP'])
best_name = best['Model']

# get the corresponding score vector we computed above
scores_map = {
    res_full['Model']: scores_full,
    res_red['Model']: scores_red,
    res_lr['Model']: scores_lr
}
valid_scores = scores_map[best_name]
valid_with_scores = valid.copy()
valid_with_scores['score'] = valid_scores

# take top 3 highest-scored dates from validation (expand to cover more transactions)
top_dates = valid_with_scores.sort_values('score', ascending=False).head(10)['date'].dt.normalize().unique()

# load raw transactions to extract consumer/supplier/cost on those dates
tx = pd.read_json("transactions.jsonl", lines=True)
tx['date'] = pd.to_datetime(tx['transaction/time'], errors='coerce').dt.normalize()

cols_keep = [
    'date',
    'transaction/consumer/id', 'transaction/consumer/name',
    'transaction/supplier/id', 'transaction/supplier/name',
    'transaction/cost'
]
tx_subset = tx.loc[tx['date'].isin(top_dates), cols_keep].rename(columns={
    'transaction/consumer/id':'consumer_id',
    'transaction/consumer/name':'consumer',
    'transaction/supplier/id':'supplier_id',
    'transaction/supplier/name':'supplier',
    'transaction/cost':'cost_sum'
})

top10 = tx_subset.sort_values('cost_sum', ascending=False).head(10)
top10[['date','consumer','supplier','cost_sum']]
